In [1]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

separators = {
    "line-productivity": "|",
    "line-downtime": "|",
    "downtime-factors": "|",
    "products": "|",
    "metadata": ",",
}

data = {}

for name, sep in separators.items():
    data[name] = pd.read_csv(f"../data/{name}.csv", sep=sep)
    print(f"--- {name} --- {data[name].shape[0]} rows, {data[name].shape[1]} columns")

--- line-productivity --- 31 rows, 6 columns
--- line-downtime --- 39 rows, 13 columns
--- downtime-factors --- 12 rows, 3 columns
--- products --- 5 rows, 4 columns
--- metadata --- 19 rows, 3 columns


In [2]:
dt = data["line-downtime"].copy()

dt = dt[dt["Batch"] != "Batch"]

print(dt.shape)
dt.head()

(38, 13)


,Batch,Factor 1,2,3,4,5,6,7,8,9,10,11,12
1,422111,NaN,60.0,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,NaN,NaN
2,422112,NaN,20.0,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,NaN,NaN
3,422113,NaN,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,422114,NaN,NaN,NaN,25.0,NaN,15.0,NaN,NaN,NaN,NaN,NaN,NaN
5,422115,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.0,NaN,NaN


In [3]:
dt.columns = ["Batch"] + list(range(1, 13))

dt = dt.reset_index(drop=True)

dt.head()

,Batch,1,2,3,4,5,6,7,8,9,10,11,12
0,422111,NaN,60.0,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,NaN,NaN
1,422112,NaN,20.0,NaN,NaN,NaN,NaN,NaN,20.0,NaN,NaN,NaN,NaN
2,422113,NaN,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,422114,NaN,NaN,NaN,25.0,NaN,15.0,NaN,NaN,NaN,NaN,NaN,NaN
4,422115,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.0,NaN,NaN


In [4]:
dt = dt.fillna(0)

dt["Batch"] = dt["Batch"].astype(int)

print(dt.dtypes)
dt.head()

Batch      int32
1        float64
2        float64
3        float64
4        float64
5        float64
6        float64
7        float64
8        float64
9        float64
10       float64
11       float64
12       float64
dtype: object


,Batch,1,2,3,4,5,6,7,8,9,10,11,12
0,422111,0.0,60.0,0.0,0.0,0.0,0.0,15.0,0.0,0.0,0.0,0.0,0.0
1,422112,0.0,20.0,0.0,0.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0
2,422113,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,422114,0.0,0.0,0.0,25.0,0.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0
4,422115,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,24.0,0.0,0.0


In [5]:
lp = data["line-productivity"].copy()

lp["End Time"] = lp["End Time"].str.replace("1900-01-01 ", "")

lp[lp["Batch"] == 422148]

,Date,Product,Batch,Operator,Start Time,End Time
30,2024-09-03,CO-2L,422148,Mac,22:55:00,01:05:00


In [6]:
lp["Start"] = pd.to_datetime(lp["Date"] + " " + lp["Start Time"])
lp["End"] = pd.to_datetime(lp["Date"] + " " + lp["End Time"])

crosses_midnight = lp["End"] < lp["Start"]
lp.loc[crosses_midnight, "End"] = lp.loc[crosses_midnight, "End"] + pd.Timedelta(days=1)

lp[lp["Batch"] == 422148][["Batch", "Start", "End"]]

,Batch,Start,End
30,422148,2024-09-03 22:55:00,2024-09-04 01:05:00


In [7]:
lp["Duration"] = (lp["End"] - lp["Start"]).dt.total_seconds() / 60

lp = lp.merge(data["products"], on="Product", how="left")

lp["Lost Time"] = lp["Duration"] - lp["Min batch time"]

lp[["Batch", "Product", "Operator", "Duration", "Min batch time", "Lost Time"]].head(10)

,Batch,Product,Operator,Duration,Min batch time,Lost Time
0,422111,OR-600,Mac,135.0,60,75.0
1,422112,LE-600,Mac,100.0,60,40.0
2,422113,LE-600,Mac,110.0,60,50.0
3,422114,LE-600,Mac,100.0,60,40.0
4,422115,LE-600,Charlie,84.0,60,24.0
5,422116,LE-600,Charlie,60.0,60,0.0
6,422117,LE-600,Charlie,75.0,60,15.0
7,422118,CO-600,Dee,120.0,60,60.0
8,422119,CO-600,Dee,85.0,60,25.0
9,422120,CO-600,Dee,112.0,60,52.0


In [8]:
dt["Total Downtime"] = dt[list(range(1, 13))].sum(axis=1)

check = lp.merge(dt[["Batch", "Total Downtime"]], on="Batch", how="outer", indicator=True)
print(check["_merge"].value_counts())

matched = check[check["_merge"] == "both"]
mismatches = matched[matched["Lost Time"] != matched["Total Downtime"]]
print("\nBatches where Lost Time != Total Downtime:", len(mismatches))

_merge
both          31
right_only     7
left_only      0
Name: count, dtype: int64

Batches where Lost Time != Total Downtime: 0


In [9]:
dt_long = dt.melt(id_vars="Batch", value_vars=list(range(1, 13)),
                  var_name="Factor", value_name="Minutes")

dt_long = dt_long[dt_long["Minutes"] > 0]

dt_long = dt_long[dt_long["Batch"].isin(lp["Batch"])]

dt_long = dt_long.merge(data["downtime-factors"], on="Factor", how="left")

print(dt_long.shape)
dt_long.sort_values("Batch").head(10)

(50, 5)


,Batch,Factor,Minutes,Description,Operator Error
0,422111,2,60.0,Batch change,Yes
25,422111,7,15.0,Machine failure,No
1,422112,2,20.0,Batch change,Yes
35,422112,8,20.0,Batch coding error,Yes
2,422113,2,50.0,Batch change,Yes
17,422114,6,15.0,Machine adjustment,Yes
6,422114,4,25.0,Inventory shortage,No
41,422115,10,24.0,Calibration error,Yes
3,422117,2,10.0,Batch change,Yes
18,422117,6,5.0,Machine adjustment,Yes


In [10]:
import os

os.makedirs("../data/processed", exist_ok=True)

batches = lp[["Batch", "Date", "Product", "Flavor", "Size", "Operator",
              "Start", "End", "Duration", "Min batch time", "Lost Time"]]

batches.to_csv("../data/processed/batches.csv", index=False)
dt_long.to_csv("../data/processed/downtime_events.csv", index=False)

print("Saved batches:", batches.shape)
print("Saved downtime events:", dt_long.shape)

Saved batches: (31, 11)
Saved downtime events: (50, 5)
